# 74. Toxicity Control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/09-adversarial/74_toxicity_control.ipynb)

**Category:** Adversarial & Safety  **Technique #:** 74  **Difficulty:** Intermediate

## Description

Toxicity control involves detecting, filtering, and preventing harmful, offensive, or inappropriate content in AI systems. This technique ensures AI applications maintain safe, respectful interactions with users across diverse contexts.

**When to use:**
- Building public-facing chatbots or conversational AI
- Creating content generation systems
- Deploying AI in educational or workplace environments
- Processing user-generated content
- Meeting platform safety requirements

## How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                    TOXICITY CONTROL                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Input ──► [Detection] ──► [Classification] ──► [Action]  │
│               │                  │               │         │
│               ▼                  ▼               ▼         │
│          Keyword/ML         Toxicity          Block/       │
│          Pattern Match      Category          Filter/      │
│                                             Warn/Allow     │
│                                                             │
│  Toxicity Categories:                                       │
│  ┌──────────┐ ┌──────────┐ ┌──────────┐ ┌──────────┐       │
│  │ Hate     │ │ Harass-  │ │ Sexually │ │ Violent  │       │
│  │ Speech   │ │ ment     │ │ Explicit │ │ Content  │       │
│  └──────────┘ └──────────┘ └──────────┘ └──────────┘       │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

**Control Strategies:**
1. **Pre-filtering** - Check inputs before processing
2. **Generation Control** - Guide model away from toxic outputs
3. **Post-filtering** - Screen outputs before delivery
4. **User Feedback Loop** - Learn from user reports
5. **Escalation Paths** - Human review for edge cases

## Setup

In [ ]:
# Install required packages
!pip install -q openai detoxify

import openai
import re
import json
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from enum import Enum
from getpass import getpass

# Import detoxify for toxicity detection
try:
    from detoxify import Detoxify
    detoxify_model = Detoxify('original')
    DETOXIFY_AVAILABLE = True
except:
    DETOXIFY_AVAILABLE = False
    print("Note: Detoxify not available. Using rule-based detection.")

# Set up OpenAI API key
openai.api_key = getpass("Enter your OpenAI API key: ")

print("✅ Setup complete!")

## Basic Example: Toxicity Detection System

In [ ]:
class ToxicityCategory(Enum):
    SAFE = "safe"
LOW = "low"
MEDIUM = "medium"
HIGH = "high"
SEVERE = "severe"

@dataclass
class ToxicityResult:
    """Result of toxicity analysis."""
    text: str
    is_toxic: bool
    toxicity_score: float
    category: ToxicityCategory
    detected_categories: Dict[str, float]
    action: str

class ToxicityDetector:
    """Multi-layer toxicity detection system."""
    
    # Toxic keywords and phrases
    TOXIC_PATTERNS = {
        'hate_speech': [
            r'\b(hate|hating)\s+\w+\s+(people|race|group)',
            r'\b(deport|kill|eliminate)\s+(all|every)\b',
            r'\b(inferior|superior)\s+(race|gender|group)\b',
        ],
        'harassment': [
            r'\b(stupid|idiot|moron|dumb)\b',
            r'\b(shut up|go away|nobody cares)\b',
            r'\b(ugly|fat|worthless|pathetic)\b',
        ],
        'threats': [
            r'\b(will|gonna|going to)\s+(kill|hurt|destroy)\b',
            r'\b(watch your back|you\'re dead|better run)\b',
        ],
        'profanity': [
            r'\b(damn|hell|crap|stupid)\b',  # Mild
        ]
    }
    
    SEVERE_KEYWORDS = [
        'kill yourself', 'kys', 'die', 'suicide',
        'rape', 'terrorist', 'bomb threat'
    ]
    
    def __init__(self, threshold: float = 0.5, use_ml: bool = True):
        """
        Initialize toxicity detector.
        
        Args:
            threshold: Toxicity score threshold (0-1)
            use_ml: Whether to use ML-based detection if available
        """
        self.threshold = threshold
        self.use_ml = use_ml and DETOXIFY_AVAILABLE
        
        # Compile patterns
        self.compiled_patterns = {}
        for category, patterns in self.TOXIC_PATTERNS.items():
            self.compiled_patterns[category] = [
                re.compile(p, re.IGNORECASE) for p in patterns
            ]
    
    def _rule_based_detect(self, text: str) -> Tuple[float, Dict[str, float]]:
        """Rule-based toxicity detection."""
        text_lower = text.lower()
        category_scores = {}
        total_score = 0.0
        
        # Check for severe keywords first
        for keyword in self.SEVERE_KEYWORDS:
            if keyword in text_lower:
                return 1.0, {'severe': 1.0}
        
        # Check each category
        for category, patterns in self.compiled_patterns.items():
            matches = sum(1 for p in patterns if p.search(text))
            score = min(matches * 0.3, 1.0)
            category_scores[category] = score
            total_score = max(total_score, score)
        
        return total_score, category_scores
    
    def _ml_detect(self, text: str) -> Tuple[float, Dict[str, float]]:
        """ML-based toxicity detection using Detoxify."""
        if not self.use_ml:
            return 0.0, {}
        
        try:
            results = detoxify_model.predict(text)
            toxicity_score = results.get('toxicity', 0.0)
            return toxicity_score, results
        except:
            return 0.0, {}
    
    def detect(self, text: str) -> ToxicityResult:
        """Detect toxicity in text."""
        # Rule-based detection
        rule_score, rule_categories = self._rule_based_detect(text)
        
        # ML-based detection
        ml_score, ml_categories = self._ml_detect(text)
        
        # Combine scores (weighted average)
        if self.use_ml:
            toxicity_score = (rule_score * 0.3) + (ml_score * 0.7)
            detected_categories = ml_categories
        else:
            toxicity_score = rule_score
            detected_categories = rule_categories
        
        # Determine category
        if toxicity_score >= 0.9:
            category = ToxicityCategory.SEVERE
            action = "block"
        elif toxicity_score >= 0.7:
            category = ToxicityCategory.HIGH
            action = "block"
        elif toxicity_score >= 0.5:
            category = ToxicityCategory.MEDIUM
            action = "warn"
        elif toxicity_score >= 0.2:
            category = ToxicityCategory.LOW
            action = "monitor"
        else:
            category = ToxicityCategory.SAFE
            action = "allow"
        
        is_toxic = toxicity_score >= self.threshold
        
        return ToxicityResult(
            text=text,
            is_toxic=is_toxic,
            toxicity_score=toxicity_score,
            category=category,
            detected_categories=detected_categories,
            action=action
        )

# Initialize detector
detector = ToxicityDetector(threshold=0.5, use_ml=DETOXIFY_AVAILABLE)

# Test cases
test_texts = [
    "Hello, how are you today?",  # Safe
    "You're such an idiot!",  # Harassment
    "I hate all people from that country",  # Hate speech
    "I'm going to hurt you",  # Threat
    "This is a great product, highly recommended!",  # Safe
]

print("=== Toxicity Detection Tests ===\n")
for text in test_texts:
    result = detector.detect(text)
    status = "🚫" if result.is_toxic else "✅"
    print(f"{status} {text}")
    print(f"   Score: {result.toxicity_score:.3f} | Category: {result.category.value} | Action: {result.action}")
    if result.detected_categories:
        print(f"   Categories: {result.detected_categories}")
    print()

## Real-World Example: Moderated Chat System

In [ ]:
class ModeratedChatSystem:
    """Chat system with comprehensive toxicity control."""
    
    SYSTEM_PROMPT = """You are a helpful assistant in a moderated chat environment.

SAFETY GUIDELINES:
- Maintain respectful and professional communication
- Do not engage with toxic or harmful content
- If a user is being inappropriate, politely redirect the conversation
- Never respond to threats or harassment with similar behavior

RESPONSE GUIDELINES:
- Keep responses helpful and constructive
- Avoid repeating or amplifying toxic content
- Model positive communication behavior
- Escalate serious issues appropriately"""
    
    RESPONSE_TEMPLATES = {
        'toxic_input': "I can't engage with that type of language. Let's keep our conversation respectful.",
        'harassment': "I'm designed to be helpful to everyone. Please keep our interaction constructive.",
        'repeated_violation': "I'm unable to continue this conversation. Please review our community guidelines.",
        'severe_toxicity': "[This message has been blocked for violating safety guidelines]"
    }
    
    def __init__(self, toxicity_threshold: float = 0.5):
        self.detector = ToxicityDetector(threshold=toxicity_threshold)
        self.conversation_history = []
        self.user_violation_count = 0
        self.max_violations = 3
        self.moderation_log = []
    
    def process_message(self, user_message: str) -> Dict:
        """Process user message with toxicity control."""
        
        # Step 1: Check input toxicity
        toxicity_result = self.detector.detect(user_message)
        
        # Log the check
        self.moderation_log.append({
            'type': 'input_check',
            'message': user_message,
            'result': toxicity_result
        })
        
        # Step 2: Handle toxic input
        if toxicity_result.category == ToxicityCategory.SEVERE:
            self.user_violation_count += 2  # Severe violations count double
            return self._handle_severe_violation(user_message, toxicity_result)
        
        elif toxicity_result.category in [ToxicityCategory.HIGH, ToxicityCategory.MEDIUM]:
            self.user_violation_count += 1
            return self._handle_moderate_violation(user_message, toxicity_result)
        
        # Step 3: Generate response for safe input
        response = self._generate_response(user_message)
        
        # Step 4: Check output toxicity
        output_check = self.detector.detect(response)
        
        if output_check.is_toxic:
            # Regenerate with stronger safety guidance
            response = self._generate_safe_response(user_message)
        
        # Update history
        self.conversation_history.append({"role": "user", "content": user_message})
        self.conversation_history.append({"role": "assistant", "content": response})
        
        return {
            'status': 'success',
            'response': response,
            'input_toxicity': toxicity_result.toxicity_score,
            'output_toxicity': output_check.toxicity_score,
            'violation_count': self.user_violation_count
        }
    
    def _handle_severe_violation(self, message: str, toxicity: ToxicityResult) -> Dict:
        """Handle severe toxicity violations."""
        if self.user_violation_count >= self.max_violations:
            return {
                'status': 'blocked',
                'response': self.RESPONSE_TEMPLATES['repeated_violation'],
                'input_toxicity': toxicity.toxicity_score,
                'action': 'conversation_ended',
                'reason': 'repeated_severe_violations'
            }
        
        return {
            'status': 'blocked',
            'response': self.RESPONSE_TEMPLATES['severe_toxicity'],
            'input_toxicity': toxicity.toxicity_score,
            'action': 'message_blocked',
            'reason': 'severe_toxicity'
        }
    
    def _handle_moderate_violation(self, message: str, toxicity: ToxicityResult) -> Dict:
        """Handle moderate toxicity violations."""
        if self.user_violation_count >= self.max_violations:
            return {
                'status': 'blocked',
                'response': self.RESPONSE_TEMPLATES['repeated_violation'],
                'input_toxicity': toxicity.toxicity_score,
                'action': 'conversation_ended',
                'reason': 'repeated_violations'
            }
        
        template_key = 'harassment' if 'harassment' in toxicity.detected_categories else 'toxic_input'
        
        return {
            'status': 'warned',
            'response': self.RESPONSE_TEMPLATES[template_key],
            'input_toxicity': toxicity.toxicity_score,
            'action': 'warning_issued',
            'reason': 'moderate_toxicity'
        }
    
    def _generate_response(self, user_message: str) -> str:
        """Generate response using LLM."""
        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            *self.conversation_history[-6:],
            {"role": "user", "content": user_message}
        ]
        
        try:
            response = openai.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=messages,
                temperature=0.5,
                max_tokens=200
            )
            return response.choices[0].message.content
        except Exception as e:
            return "I apologize, but I'm having trouble processing your request."
    
    def _generate_safe_response(self, user_message: str) -> str:
        """Generate response with enhanced safety."""
        safe_prompt = f"""Provide a helpful, safe response to: {user_message}

Important: Keep the response respectful, constructive, and free of any harmful content."""
        
        messages = [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": safe_prompt}
        ]
        
        try:
            response = openai.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=messages,
                temperature=0.3,
                max_tokens=200
            )
            return response.choices[0].message.content
        except:
            return "I'm here to help with your questions. What would you like to know?"
    
    def get_moderation_stats(self) -> Dict:
        """Get moderation statistics."""
        total_checks = len(self.moderation_log)
        violations = sum(1 for log in self.moderation_log if log['result'].is_toxic)
        
        return {
            'total_messages_checked': total_checks,
            'violations_detected': violations,
            'violation_rate': violations / total_checks * 100 if total_checks > 0 else 0,
            'current_violation_count': self.user_violation_count,
            'max_violations_allowed': self.max_violations
        }

# Initialize moderated chat
chat = ModeratedChatSystem(toxicity_threshold=0.5)

# Test scenarios
test_messages = [
    "Hello, can you help me with a question?",
    "You're so stupid and useless!",
    "What's the weather like today?",
    "I hate people like you",
    "Can you explain quantum physics?"
]

print("=== Moderated Chat System Tests ===\n")
for msg in test_messages:
    print(f"User: {msg}")
    result = chat.process_message(msg)
    print(f"Status: {result['status']}")
    print(f"Response: {result['response'][:80]}...")
    print(f"Toxicity Score: {result.get('input_toxicity', 0):.3f}")
    print("-" * 50 + "\n")

# Show stats
stats = chat.get_moderation_stats()
print("\n=== Moderation Statistics ===")
print(f"Messages Checked: {stats['total_messages_checked']}")
print(f"Violations: {stats['violations_detected']}")
print(f"Violation Rate: {stats['violation_rate']:.1f}%")

## Failure Case: Toxicity Detection Limitations

In [ ]:
# Demonstrate toxicity detection limitations

limitation_examples = [
    {
        'name': 'Context-Dependent Toxicity',
        'text': "You're killing it!",
        'issue': 'Positive slang that contains toxic words',
        'correct_interpretation': 'Compliment (doing great)',
        'detection_challenge': 'High - may false positive'
    },
    {
        'name': 'Reclaimed Language',
        'text': "We're taking back this word for our community",
        'issue': 'Words acceptable within a group but not from outsiders',
        'correct_interpretation': 'Context-dependent acceptability',
        'detection_challenge': 'Very High - requires cultural knowledge'
    },
    {
        'name': 'Sarcasm and Irony',
        'text': "Oh great, another brilliant idea from the genius",
        'issue': 'Negative sentiment disguised as positive language',
        'correct_interpretation': 'Sarcastic criticism',
        'detection_challenge': 'Very High - requires tone understanding'
    },
    {
        'name': 'Code Switching',
        'text': "This is some good ish right here",
        'issue': 'Informal/dialect language that may trigger filters',
        'correct_interpretation': 'Positive informal expression',
        'detection_challenge': 'High - dialect bias in training data'
    },
    {
        'name': 'Subtle Toxicity',
        'text': "I'm sure you did your best, given your background",
        'issue': 'Condescension without explicit toxic words',
        'correct_interpretation': 'Passive-aggressive insult',
        'detection_challenge': 'Very High - requires deep semantic analysis'
    },
    {
        'name': 'Euphemisms',
        'text': "Those people always stick together",
        'issue': 'Coded language for discrimination',
        'correct_interpretation': 'Potentially prejudiced statement',
        'detection_challenge': 'Very High - indirect expression'
    }
]

print("=== Toxicity Detection Limitations ===\n")
for example in limitation_examples:
    result = detector.detect(example['text'])
    detected = "✅ Detected" if result.is_toxic else "❌ Not Detected"
    
    print(f"🔍 {example['name']}")
    print(f"   Text: '{example['text']}'")
    print(f"   Issue: {example['issue']}")
    print(f"   Detection: {detected} (score: {result.toxicity_score:.3f})")
    print(f"   Challenge: {example['detection_challenge']}")
    print()

print("\n💡 Mitigation Strategies:")
print("  1. Context-aware detection (conversation history)")
print("  2. User preference settings for sensitivity")
print("  3. Human review for edge cases")
print("  4. Cultural and dialect training data")
print("  5. Sarcasm and irony detection models")
print("  6. User feedback integration")
print("  7. Graduated response system (warn vs block)")

## Benchmark: Toxicity Detection Performance

In [ ]:
import pandas as pd

# Detection method comparison
detection_methods = {
    'Method': [
        'Keyword Matching',
        'Regex Patterns',
        'ML Classifier (Basic)',
        'ML Classifier (BERT)',
        'OpenAI Moderation',
        'Perspective API',
        'Detoxify',
        'Ensemble (Multiple)'
    ],
    'Precision': ['75%', '80%', '82%', '88%', '90%', '85%', '87%', '92%'],
    'Recall': ['70%', '75%', '85%', '90%', '88%', '86%', '89%', '93%'],
    'F1 Score': ['0.72', '0.77', '0.83', '0.89', '0.89', '0.85', '0.88', '0.92'],
    'Speed': ['Very Fast', 'Fast', 'Fast', 'Medium', 'Medium', 'Medium', 'Fast', 'Slow'],
    'Cost': ['Free', 'Free', 'Low', 'Medium', 'API Cost', 'API Cost', 'Free', 'High']
}

df = pd.DataFrame(detection_methods)
print("=== Toxicity Detection Methods ===\n")
print(df.to_string(index=False))

# Toxicity category detection rates
print("\n\n=== Detection Rates by Category ===\n")

category_rates = {
    'Category': [
        'Hate Speech',
        'Harassment',
        'Threats',
        'Profanity (Mild)',
        'Profanity (Severe)',
        'Sexual Content',
        'Self-Harm',
        'Violence',
        'Spam',
        'Sarcasm/Trolling'
    ],
    'Detection Rate': ['85%', '80%', '92%', '95%', '98%', '90%', '88%', '87%', '75%', '45%'],
    'False Positive Risk': ['Medium', 'Medium', 'Low', 'High', 'Low', 'Low', 'Low', 'Medium', 'Low', 'Very High']
}

df_cat = pd.DataFrame(category_rates)
print(df_cat.to_string(index=False))

## Interactive Playground

In [ ]:
# Interactive toxicity testing

def interactive_toxicity_test():
    """Interactive toxicity detection testing."""
    print("=== Toxicity Control Playground ===\n")
    print("Test text for toxicity. Type 'quit' to exit.\n")
    
    # Example test texts
    examples = [
        "I love this product!",
        "This is terrible service",
        "You're doing great work",
        "Shut up and go away",
        "I disagree with your opinion",
        "That's the stupidest thing I've ever heard"
    ]
    
    print("Example texts to try:")
    for i, ex in enumerate(examples, 1):
        print(f"  {i}. {ex}")
    
    print("\nOr enter your own text to test.\n")
    
    while True:
        user_input = input("Test text: ")
        
        if user_input.lower() == 'quit':
            break
        
        result = detector.detect(user_input)
        
        # Visual indicator
        if result.category == ToxicityCategory.SEVERE:
            icon = "🔴"
        elif result.category == ToxicityCategory.HIGH:
            icon = "🟠"
        elif result.category == ToxicityCategory.MEDIUM:
            icon = "🟡"
        elif result.category == ToxicityCategory.LOW:
            icon = "🟡"
        else:
            icon = "🟢"
        
        print(f"\n{icon} Results:")
        print(f"   Toxicity Score: {result.toxicity_score:.3f}/1.0")
        print(f"   Category: {result.category.value.upper()}")
        print(f"   Is Toxic: {result.is_toxic}")
        print(f"   Recommended Action: {result.action}")
        
        if result.detected_categories:
            print(f"   Detected Categories:")
            for cat, score in result.detected_categories.items():
                if isinstance(score, float):
                    print(f"      - {cat}: {score:.3f}")
                else:
                    print(f"      - {cat}")
        
        print()

# Run interactive test (uncomment to use)
# interactive_toxicity_test()

# Demo with pre-loaded examples
print("=== Pre-loaded Example Analysis ===\n")

demo_texts = [
    ("Safe", "Thank you for your help!"),
    ("Mild", "This is really frustrating"),
    ("Medium", "You're being completely unreasonable"),
    ("High", "I hate dealing with people like you"),
    ("Severe", "You should just disappear forever")
]

for expected, text in demo_texts:
    result = detector.detect(text)
    icon = "🚫" if result.is_toxic else "✅"
    print(f"{icon} [{expected}] {text}")
    print(f"   Score: {result.toxicity_score:.3f} | Category: {result.category.value} | Action: {result.action}\n")

## Tips & Tricks

### Model-Specific Recommendations

**OpenAI GPT Models:**
- Use OpenAI's `moderation` endpoint as primary filter
- Set appropriate thresholds per category
- Consider using `gpt-4` for better safety adherence
- Lower temperature reduces toxic output generation

**Anthropic Claude:**
- Leverage Claude's constitutional AI training
- Generally produces less toxic outputs
- Still implement output filtering as backup

**Google Gemini:**
- Configure `safety_settings` appropriately
- Use `HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE` for production
- Test with all harm categories enabled

### Best Practices

1. **Layered Defense**: Use multiple detection methods
2. **Graduated Response**: Warn before blocking
3. **User Feedback**: Allow users to report false positives
4. **Regular Updates**: Update detection patterns frequently
5. **Context Awareness**: Consider conversation history
6. **Transparency**: Explain why content was blocked

### Response Strategy

| Toxicity Level | Action | Response Type |
|----------------|--------|---------------|
| Safe (0-0.2) | Allow | Normal response |
| Low (0.2-0.5) | Monitor | Normal response |
| Medium (0.5-0.7) | Warn | Warning + redirect |
| High (0.7-0.9) | Block | Block message |
| Severe (0.9+) | Block + Log | Immediate block |

## References

1. **Hanu, L., & Detoxify Team. (2020).** "Detoxify: Machine Learning Model for Toxicity Detection." https://github.com/unitaryai/detoxify

2. **OpenAI. (2024).** "Moderation API Documentation." https://platform.openai.com/docs/guides/moderation

3. **Perspective API. (2024).** "Toxicity Detection API." https://perspectiveapi.com/

4. **Wulczyn, E., et al. (2017).** "Ex Machina: Personal Attacks Seen at Scale." *WWW 2017*.

5. **Google Jigsaw. (2024).** "Conversation AI." https://jigsaw.google.com/

6. **OWASP. (2024).** "LLM03: Insecure Output Handling." https://owasp.org/www-project-top-10-for-large-language-model-applications/